# Exercise XP Gold: Run Code Llama on Kaggle using Gemma
Author: arielzin33@gmail.com

**Model path chosen:** Gemma 3 1B Instruct (GGUF, Q8_0) — the exact public model given in the exercise's own snippet. It requires **no Kaggle access request** and is already in GGUF format, so no Hugging Face → GGUF conversion step is needed. A clearly marked **optional Part 2B** below shows the Code Llama 7B path (convert-hf-to-gguf.py) for if/when your Kaggle access request for `metaresearch/codellama/PyTorch/7b-python-hf` is approved — swap `MODEL_PATH` to point at that converted file and every downstream cell (Part 3 onward) works unchanged.

**Runtime:** run this in a **Kaggle notebook with GPU (T4 x2 or P100) enabled**, or Google Colab with a GPU runtime (`Runtime → Change runtime type → GPU`).

---
## Part 1: Install `llama-cpp-python` with CUDA bindings and clone `llama.cpp`


In [ ]:
# CUDA-enabled build of llama-cpp-python so inference offloads layers to the GPU.
# CMAKE_ARGS tells the build to compile the CUDA backend (cuBLAS); -DGGML_CUDA=on is the current flag name
# for recent llama.cpp/llama-cpp-python releases (older versions used -DLLAMA_CUBLAS=on).
import os
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

!pip install llama-cpp-python --verbose


In [ ]:
# Clone llama.cpp — used here mainly for reference (conversion scripts, quantization tools),
# since llama-cpp-python already bundles a compatible inference engine.
!git clone https://github.com/ggerganov/llama.cpp.git


---
## Part 2: Model Preparation


### Part 2A — Gemma 3 1B GGUF (public, no access request, no conversion needed)

This is already a GGUF file, so there is nothing to convert — download it directly and it's immediately loadable by `llama-cpp-python`.

In [ ]:
!wget https://huggingface.co/MaziyarPanahi/gemma-3-1b-it-GGUF/resolve/main/gemma-3-1b-it.Q8_0.gguf \
    -O gemma-3-1b-it-Q8_0.gguf

MODEL_PATH = "gemma-3-1b-it-Q8_0.gguf"


### Part 2B — Code Llama 7B (optional — only if your Kaggle access is approved)

1. On the [Code Llama 7B Kaggle model page](https://www.kaggle.com/models/metaresearch/codellama/PyTorch/7b-python-hf/1), click **Request Access**; once approved, add the model as a Kaggle notebook **Input** (or download it via the Kaggle API) so its weights appear under `/kaggle/input/...`.
2. Convert the downloaded Hugging Face–format weights to GGUF using the script from the `llama.cpp` repo cloned in Part 1:


In [ ]:
# Only run this cell if you have approved Kaggle access to Code Llama 7B and have added it
# as a notebook input (path will look like /kaggle/input/codellama-7b-python-hf/...).

CODELLAMA_HF_PATH = "/kaggle/input/codellama-7b-python-hf"  # adjust to your actual input path

!python3 llama.cpp/convert_hf_to_gguf.py \
    --model {CODELLAMA_HF_PATH} \
    --outfile codellama-7b.gguf

# To use this model for the rest of the notebook instead of Gemma, uncomment:
# MODEL_PATH = "codellama-7b.gguf"


*Note:* the conversion script in `llama.cpp` was renamed from `convert-hf-to-gguf.py` to `convert_hf_to_gguf.py` (hyphens → underscores) in more recent versions of the repo — if the hyphenated filename from the assignment prompt errors with "file not found", use the underscored one shown above (or run `ls llama.cpp/convert*` to confirm the exact filename in your clone).

---
## Part 3: Load the Model and Generate Text/Code


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,   # -1 offloads all layers to GPU; reduce (e.g., 20) if you hit out-of-memory errors
    n_threads=4,        # CPU threads for any non-offloaded work
    n_ctx=2048,          # context window in tokens
    verbose=False,
)


### 3a. Natural-language prompt

In [ ]:
prompt = "Explain how the solar system formed."

output = llm(prompt=prompt, max_tokens=256)
print(output["choices"][0]["text"])


### 3b. Code-generation prompt, with streaming output

In [ ]:
code_prompt = "Write a Python script that loads a Hugging Face model and tokenizes input."

output_stream = llm(
    prompt=code_prompt,
    max_tokens=512,
    stream=True,  # Enable streaming
)

# Iterate through the tokens and print them as they are computed
for chunk in output_stream:
    token = chunk["choices"][0]["text"]
    print(token, end="", flush=True)


### Filled-in hint block (for reference — matches the assignment's fill-in-the-blank snippet)

```python
from llama_cpp import Llama

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_threads=4,
)

prompt = "Explain how the solar system formed."

output = llm(prompt=prompt, max_tokens=256)
print(output["choices"][0]["text"])

output_stream = llm(
    prompt="Write a Python script that loads a Hugging Face model and tokenizes input.",
    max_tokens=512,
    stream=True,  # Enable streaming
)

for output in output_stream:
    token = output["choices"][0]["text"]
    print(token, end="", flush=True)
```

---
## 🎯 Bonus Exercises

A small `ask()` helper so each bonus prompt is a one-liner, followed by the 5 requested prompts.

In [ ]:
def ask(prompt, max_tokens=400):
    result = llm(prompt=prompt, max_tokens=max_tokens)
    print(result["choices"][0]["text"])


**1. Prime-number checker**

In [ ]:
ask("Write a Python function that checks if a number is prime.")


**2. CSV line chart with matplotlib**

In [ ]:
ask("Write a Python script that reads a CSV file and plots a line chart using matplotlib.")


**3. Headline scraper with requests + BeautifulSoup**

In [ ]:
ask("Write a simple Python web scraper that extracts all headlines from a news site using "
    "requests and BeautifulSoup.")


**4. List vs. tuple explanation**

In [ ]:
ask("Explain the difference between a list and a tuple in Python.")


**5. Even numbers for-loop**

In [ ]:
ask("Write a for-loop that prints even numbers between 1 and 100.")


---
## Troubleshooting notes

- **Out of GPU memory:** lower `n_gpu_layers` from `-1` to a fixed number (e.g., `20`) so only part of the model offloads to GPU, or switch to a smaller quantization (e.g., `Q4_K_M` instead of `Q8_0`).
- **`pip install llama-cpp-python` doesn't pick up CUDA:** confirm the Kaggle/Colab GPU runtime is actually enabled (`!nvidia-smi` should show a GPU) *before* running the install cell — the `CMAKE_ARGS`/`FORCE_CMAKE` env vars must be set before `pip install`, not after.
- **Conversion script name mismatch:** run `!ls llama.cpp/convert*` to see the exact filename in your cloned version if `convert-hf-to-gguf.py` or `convert_hf_to_gguf.py` isn't found — the repo has renamed this script across versions.